# NSE Stocks + NIFTY 50 Incremental Historical Sync

**Filesystem-first design**

- Existing Parquet files are authoritative.
- Existing stock/NIFTY Parquet files are **never requested again**.
- A previously failed/error date is retried automatically **only when its Parquet file is still missing**.
- The manifest is diagnostic only; it never decides whether a date is downloadable.
- There is a hard preflight check before any network request.
- There is another filesystem check immediately before every network request.
- NIFTY 50 uses the official `niftyindices.com` historical-data API discovered from the site's JavaScript.
- NIFTY API responses are locally filtered because the endpoint can return records outside the requested date range.
- The notebook is safe to rerun.

Run the cells top-to-bottom.


In [ ]:
# ============================================================
# 1. SETUP / IMPORTS
# ============================================================

import os
import re
import json
import time
import random
import hashlib
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

print("Notebook version: NSE_STOCKS_NIFTY50_SYNC_v3")
print("Started:", datetime.now().isoformat(timespec="seconds"))


In [ ]:
# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

BASE_DIR = Path("/content/drive/MyDrive/quant")
STOCK_DIR = BASE_DIR / "data" / "parquet"
NIFTY_DIR = BASE_DIR / "data" / "indices" / "nifty50"
MANIFEST_DIR = BASE_DIR / "data" / "manifests"

STOCK_DIR.mkdir(parents=True, exist_ok=True)
NIFTY_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR     :", BASE_DIR)
print("STOCK_DIR    :", STOCK_DIR)
print("NIFTY_DIR    :", NIFTY_DIR)
print("MANIFEST_DIR :", MANIFEST_DIR)


In [ ]:
# ============================================================
# 3. CONFIGURATION
# ============================================================

START_DATE = date(2011, 1, 1)
END_DATE = date.today()

DOWNLOAD_STOCKS = True
DOWNLOAD_NIFTY = True

# NIFTY API chunk size. The endpoint may return extra records,
# so every response is filtered locally to the requested range.
NIFTY_CHUNK_DAYS = 120

REQUEST_TIMEOUT = 60
MAX_RETRIES = 4
RETRY_BASE_SECONDS = 2

STOCK_MANIFEST = MANIFEST_DIR / "nse_stock_download_manifest.csv"
NIFTY_MANIFEST = MANIFEST_DIR / "nifty50_download_manifest.csv"

print("Date range:", START_DATE, "->", END_DATE)


In [ ]:
# ============================================================
# 4. COMMON HELPERS
# ============================================================

def date_range_days(start, end):
    cur = start
    while cur <= end:
        yield cur
        cur += timedelta(days=1)

def parquet_path(directory, day):
    return directory / f"{day.isoformat()}.parquet"

def atomic_to_parquet(df, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    if tmp.exists():
        tmp.unlink()
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)

def append_manifest(row, manifest_path):
    row = dict(row)
    row.setdefault("timestamp", datetime.now().isoformat(timespec="seconds"))

    df = pd.DataFrame([row])

    if manifest_path.exists():
        df.to_csv(
            manifest_path,
            mode="a",
            header=False,
            index=False,
        )
    else:
        df.to_csv(
            manifest_path,
            mode="w",
            header=True,
            index=False,
        )

def existing_parquet_dates(directory, start, end):
    found = []
    for day in date_range_days(start, end):
        if parquet_path(directory, day).exists():
            found.append(day)
    return found

def print_inventory(name, existing, missing):
    print(f"\n{name}")
    print("-" * 70)
    print(f"Existing Parquet : {len(existing):,}")
    print(f"Missing Parquet  : {len(missing):,}")
    if existing:
        print(f"Existing range   : {min(existing)} -> {max(existing)}")
    if missing:
        print(f"Missing range    : {min(missing)} -> {max(missing)}")

def is_weekend(day):
    return day.weekday() >= 5


## 5. STOCK NSE BHA VCOPY SOURCES

Legacy source through **2024-07-05**:

`https://nsearchives.nseindia.com/content/historical/EQUITIES/{year}/{MONTH}/cm{DD}{MON}{YYYY}bhav.csv.zip`

UDiFF source from **2024-07-08**:

`https://nsearchives.nseindia.com/content/cm/BhavCopy_NSE_CM_0_0_0_{YYYYMMDD}_F_0000.csv.zip`

The filesystem check is performed before every request.


In [ ]:
# ============================================================
# 6. NSE STOCK SESSION + URL BUILDERS
# ============================================================

NSE_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    ),
    "Accept": "*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/",
    "Connection": "keep-alive",
}

def make_nse_session():
    s = requests.Session()
    s.headers.update(NSE_HEADERS)

    # Warm-up request helps establish NSE cookies/session.
    try:
        s.get(
            "https://www.nseindia.com/",
            timeout=REQUEST_TIMEOUT,
        )
    except Exception as e:
        print("NSE warm-up warning:", repr(e))

    return s

nse_session = make_nse_session()

def legacy_stock_url(day):
    month = day.strftime("%b").upper()
    return (
        "https://nsearchives.nseindia.com/content/historical/EQUITIES/"
        f"{day.year}/{month}/"
        f"cm{day.strftime('%d')}{month}{day.strftime('%Y')}bhav.csv.zip"
    )

def udiff_stock_url(day):
    return (
        "https://nsearchives.nseindia.com/content/cm/"
        f"BhavCopy_NSE_CM_0_0_0_{day.strftime('%Y%m%d')}_F_0000.csv.zip"
    )

def stock_url(day):
    # NSE UDiFF started on 2024-07-08.
    if day <= date(2024, 7, 5):
        return legacy_stock_url(day)
    return udiff_stock_url(day)


In [ ]:
# ============================================================
# 7. STOCK PARSER
# ============================================================

def parse_stock_bhavcopy(content):
    import io
    import zipfile

    with zipfile.ZipFile(io.BytesIO(content)) as z:
        names = z.namelist()
        csv_names = [n for n in names if n.lower().endswith(".csv")]

        if not csv_names:
            raise ValueError(f"No CSV found inside ZIP: {names}")

        with z.open(csv_names[0]) as f:
            raw = pd.read_csv(f)

    raw.columns = [str(c).strip().upper() for c in raw.columns]

    # Handle common NSE column names.
    symbol_col = next(
        (c for c in ["SYMBOL", "TckrSymb", "TICKER"] if c in raw.columns),
        None,
    )

    open_col = next(
        (c for c in ["OPEN", "OpnPric"] if c in raw.columns),
        None,
    )
    high_col = next(
        (c for c in ["HIGH", "HghPric"] if c in raw.columns),
        None,
    )
    low_col = next(
        (c for c in ["LOW", "LwPric"] if c in raw.columns),
        None,
    )
    close_col = next(
        (c for c in ["CLOSE", "ClsPric"] if c in raw.columns),
        None,
    )
    volume_col = next(
        (c for c in ["TOTTRDQTY", "TTL_TRD_QNTY", "TOTTRDQUANTITY", "TotTrdQty"] if c in raw.columns),
        None,
    )

    required = {
        "symbol": symbol_col,
        "open": open_col,
        "high": high_col,
        "low": low_col,
        "close": close_col,
        "volume": volume_col,
    }

    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(
            f"Could not identify stock columns: {missing}. "
            f"Available columns: {list(raw.columns)}"
        )

    out = pd.DataFrame({
        "date": pd.NaT,
        "symbol": raw[symbol_col].astype(str).str.strip(),
        "open": pd.to_numeric(raw[open_col], errors="coerce"),
        "high": pd.to_numeric(raw[high_col], errors="coerce"),
        "low": pd.to_numeric(raw[low_col], errors="coerce"),
        "close": pd.to_numeric(raw[close_col], errors="coerce"),
        "volume": pd.to_numeric(raw[volume_col], errors="coerce"),
    })

    out = out.dropna(subset=["symbol", "open", "high", "low", "close"])
    return out


In [ ]:
# ============================================================
# 8. DOWNLOAD ONE STOCK DAY
# ============================================================

def download_stock_day(day):
    path = parquet_path(STOCK_DIR, day)

    # --------------------------------------------------------
    # HARD RULE: never request an existing file
    # --------------------------------------------------------
    if path.exists():
        return {
            "date": day,
            "status": "skipped_existing",
            "rows": 0,
            "url": "",
            "error": "",
        }

    url = stock_url(day)

    for attempt in range(1, MAX_RETRIES + 1):

        # Re-check immediately before EVERY network request.
        if path.exists():
            return {
                "date": day,
                "status": "skipped_existing",
                "rows": 0,
                "url": url,
                "error": "",
            }

        try:
            response = nse_session.get(
                url,
                timeout=REQUEST_TIMEOUT,
            )

            if response.status_code == 404:
                return {
                    "date": day,
                    "status": "not_available",
                    "rows": 0,
                    "url": url,
                    "error": "HTTP 404",
                }

            response.raise_for_status()

            df = parse_stock_bhavcopy(response.content)

            if df.empty:
                raise ValueError("Parsed stock dataframe is empty")

            df["date"] = pd.Timestamp(day)

            # Final check before writing.
            if path.exists():
                return {
                    "date": day,
                    "status": "skipped_existing",
                    "rows": 0,
                    "url": url,
                    "error": "",
                }

            atomic_to_parquet(df, path)

            return {
                "date": day,
                "status": "downloaded",
                "rows": len(df),
                "url": url,
                "error": "",
            }

        except Exception as e:
            if attempt >= MAX_RETRIES:
                return {
                    "date": day,
                    "status": "failed",
                    "rows": 0,
                    "url": url,
                    "error": repr(e),
                }

            sleep_for = RETRY_BASE_SECONDS * (2 ** (attempt - 1))
            sleep_for += random.uniform(0, 1)
            time.sleep(sleep_for)

    raise RuntimeError("Unreachable")


In [ ]:
# ============================================================
# 9. STOCK FILESYSTEM INVENTORY
# ============================================================
# IMPORTANT:
# The manifest is NOT consulted here.
# Filesystem existence alone determines whether a date is queued.

stock_candidates = list(date_range_days(START_DATE, END_DATE))

stock_existing = []
stock_missing = []

for day in stock_candidates:
    path = parquet_path(STOCK_DIR, day)

    if path.exists():
        stock_existing.append(day)
    else:
        stock_missing.append(day)

# ONLY missing Parquet files are queued.
stock_download_queue = list(stock_missing)

print_inventory(
    "STOCKS",
    stock_existing,
    stock_missing,
)

print("\nStock queue:", len(stock_download_queue))


In [ ]:
# ============================================================
# 10. STOCK PREFLIGHT
# ============================================================

# Hard guarantee: no queued date currently has a Parquet file.
stock_existing_before_download = [
    day
    for day in stock_download_queue
    if parquet_path(STOCK_DIR, day).exists()
]

assert not stock_existing_before_download, (
    "ABORT: stock queue contains existing files: "
    f"{stock_existing_before_download[:20]}"
)

print("STOCK PREFLIGHT PASSED")
print("Queued missing files:", len(stock_download_queue))
print("Existing queued files:", len(stock_existing_before_download))


In [ ]:
# ============================================================
# 11. STOCK DOWNLOAD LOOP
# ============================================================

stock_results = []

if DOWNLOAD_STOCKS and stock_download_queue:

    print(
        f"Downloading {len(stock_download_queue):,} missing stock dates. "
        "Existing files will be skipped."
    )

    for day in tqdm(
        stock_download_queue,
        desc="NSE missing stock files",
    ):
        path = parquet_path(STOCK_DIR, day)

        # Safety check immediately before this date's request.
        if path.exists():
            result = {
                "date": day,
                "status": "skipped_existing",
                "rows": 0,
                "url": "",
                "error": "",
            }
        else:
            result = download_stock_day(day)

        stock_results.append(result)
        append_manifest(result, STOCK_MANIFEST)

else:
    print("No missing stock files. No stock network requests will be made.")

stock_results_df = pd.DataFrame(stock_results)


In [ ]:
# ============================================================
# 12. STOCK SUMMARY
# ============================================================

if stock_results_df.empty:
    print("No stock downloads attempted.")
else:
    print("\nSTOCK SYNC SUMMARY")
    print(stock_results_df["status"].value_counts(dropna=False))

    failed = stock_results_df[
        stock_results_df["status"].isin(["failed"])
    ]

    if not failed.empty:
        print("\nFailed stock dates:")
        display(failed[["date", "status", "error"]].head(100))

    print("\nDownloaded rows:", int(
        pd.to_numeric(
            stock_results_df.get("rows", pd.Series(dtype=float)),
            errors="coerce"
        ).fillna(0).sum()
    ))


# NIFTY 50

The NIFTY API is the official historical-data endpoint used by the NIFTY Indices site:

`https://www.niftyindices.com/BackPage/getHistoricaldatatabletoString`

The request payload must contain `cinfo` with the historical request encoded as the endpoint expects.

Important: this endpoint can return records outside the requested end date. Therefore **every response is filtered locally** before anything is written.


In [ ]:
# ============================================================
# 13. NIFTY API CONFIG
# ============================================================

NIFTY_ENDPOINT = (
    "https://www.niftyindices.com/"
    "BackPage/getHistoricaldatatabletoString"
)

NIFTY_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Content-Type": "application/json; charset=utf-8",
    "Origin": "https://www.niftyindices.com",
    "Referer": "https://www.niftyindices.com/reports",
    "X-Requested-With": "XMLHttpRequest",
}

def make_nifty_session():
    s = requests.Session()
    s.headers.update(NIFTY_HEADERS)

    # Warm up the site to establish cookies.
    try:
        s.get(
            "https://www.niftyindices.com/reports",
            timeout=REQUEST_TIMEOUT,
        )
    except Exception as e:
        print("NIFTY warm-up warning:", repr(e))

    return s

nifty_session = make_nifty_session()

print("NIFTY endpoint:", NIFTY_ENDPOINT)


In [ ]:
# ============================================================
# 14. NIFTY API REQUEST / PARSER
# ============================================================

def make_nifty_payload(start_day, end_day):
    # This mirrors the payload structure used by the official site's JS.
    cinfo = (
        "{'name':'NIFTY 50',"
        f"'startDate':'{start_day.strftime('%d-%m-%Y')}',"
        f"'endDate':'{end_day.strftime('%d-%m-%Y')}',"
        "'indexName':'NIFTY 50'}"
    )

    return {
        "cinfo": cinfo
    }


def normalize_nifty_records(records):
    if not isinstance(records, list):
        raise ValueError(
            f"Expected list from NIFTY API, got {type(records).__name__}"
        )

    raw = pd.DataFrame(records)

    if raw.empty:
        return pd.DataFrame(
            columns=["date", "symbol", "open", "high", "low", "close", "volume"]
        )

    # Print columns once per response if useful.
    date_col = next(
        (
            c for c in raw.columns
            if str(c).strip().lower() in {
                "historicaldate",
                "historical date",
                "date",
                "indexdate",
            }
        ),
        None,
    )

    open_col = next(
        (
            c for c in raw.columns
            if str(c).strip().lower() in {
                "open",
                "openprice",
                "open price",
            }
        ),
        None,
    )

    high_col = next(
        (
            c for c in raw.columns
            if str(c).strip().lower() in {
                "high",
                "highprice",
                "high price",
            }
        ),
        None,
    )

    low_col = next(
        (
            c for c in raw.columns
            if str(c).strip().lower() in {
                "low",
                "lowprice",
                "low price",
            }
        ),
        None,
    )

    close_col = next(
        (
            c for c in raw.columns
            if str(c).strip().lower() in {
                "close",
                "closingindex",
                "closing index",
                "closeprice",
            }
        ),
        None,
    )

    if not all([date_col, open_col, high_col, low_col, close_col]):
        raise ValueError(
            "Could not identify NIFTY fields. "
            f"Returned columns: {list(raw.columns)}"
        )

    out = pd.DataFrame({
        "date": pd.to_datetime(
            raw[date_col],
            errors="coerce",
            dayfirst=True,
        ).dt.date,
        "symbol": "NIFTY50",
        "open": pd.to_numeric(raw[open_col], errors="coerce"),
        "high": pd.to_numeric(raw[high_col], errors="coerce"),
        "low": pd.to_numeric(raw[low_col], errors="coerce"),
        "close": pd.to_numeric(raw[close_col], errors="coerce"),
        "volume": np.nan,
    })

    out = out.dropna(
        subset=["date", "open", "high", "low", "close"]
    )

    return out


In [ ]:
# ============================================================
# 15. NIFTY FETCH WITH PROPER OFFICIAL API
# ============================================================

def fetch_nifty_range(session, start_day, end_day, verbose=True):

    payload = make_nifty_payload(start_day, end_day)

    for attempt in range(1, MAX_RETRIES + 1):

        try:
            response = session.post(
                NIFTY_ENDPOINT,
                headers=NIFTY_HEADERS,
                json=payload,
                timeout=REQUEST_TIMEOUT,
            )

            if verbose:
                print(
                    f"NIFTY request {start_day} -> {end_day} | "
                    f"attempt={attempt} | "
                    f"HTTP={response.status_code} | "
                    f"Content-Type={response.headers.get('Content-Type')} | "
                    f"bytes={len(response.content):,}"
                )

            response.raise_for_status()

            # The endpoint has historically returned JSON while
            # sometimes declaring text/html. Therefore parse the body,
            # not the Content-Type.
            try:
                records = response.json()
            except Exception:
                text_body = response.text.strip()

                # Some responses may contain a JSON-looking body despite
                # unusual formatting/content-type.
                records = json.loads(text_body)

            df = normalize_nifty_records(records)

            if df.empty:
                print(
                    f"NIFTY API returned no usable rows for "
                    f"{start_day} -> {end_day}"
                )
                return df

            actual_min = min(df["date"])
            actual_max = max(df["date"])

            if verbose:
                print(
                    f"  API returned date range: "
                    f"{actual_min} -> {actual_max}; "
                    f"rows={len(df):,}"
                )

            # CRITICAL: endpoint may return dates outside the request.
            before = len(df)

            df = df[
                (df["date"] >= start_day) &
                (df["date"] <= end_day)
            ].copy()

            outside = before - len(df)

            if verbose and outside:
                print(
                    f"  Filtered {outside:,} out-of-range records locally."
                )

            if not df.empty:
                assert df["date"].min() >= start_day
                assert df["date"].max() <= end_day

            return df

        except Exception as e:
            if attempt >= MAX_RETRIES:
                raise RuntimeError(
                    f"NIFTY API failed after {MAX_RETRIES} attempts "
                    f"for {start_day} -> {end_day}: {repr(e)}"
                ) from e

            sleep_for = RETRY_BASE_SECONDS * (2 ** (attempt - 1))
            sleep_for += random.uniform(0, 1)

            print(
                f"  NIFTY request failed: {repr(e)}; "
                f"retrying in {sleep_for:.1f}s"
            )

            time.sleep(sleep_for)


In [ ]:
# ============================================================
# 16. NIFTY API DIAGNOSTIC TEST
# ============================================================
# Run this before the full NIFTY sync.
# It proves the official API/payload works and that local date
# filtering is active.

TEST_START = date(2025, 1, 1)
TEST_END = date(2025, 1, 10)

test_nifty = fetch_nifty_range(
    nifty_session,
    TEST_START,
    TEST_END,
    verbose=True,
)

print("\nNIFTY TEST RESULT")
print("-" * 70)
print("Rows in requested range:", len(test_nifty))

if not test_nifty.empty:
    print("First date:", min(test_nifty["date"]))
    print("Last date :", max(test_nifty["date"]))
    display(test_nifty)
else:
    print("WARNING: API returned no rows for test range.")


In [ ]:
# ============================================================
# 17. NIFTY FILESYSTEM INVENTORY
# ============================================================
# Filesystem is authoritative. Manifest is ignored for queueing.

nifty_candidates = list(date_range_days(START_DATE, END_DATE))

nifty_existing = []
nifty_missing = []

for day in nifty_candidates:
    path = parquet_path(NIFTY_DIR, day)

    if path.exists():
        nifty_existing.append(day)
    else:
        nifty_missing.append(day)

# ONLY missing files are queued.
nifty_download_queue = list(nifty_missing)

print_inventory(
    "NIFTY 50",
    nifty_existing,
    nifty_missing,
)

print("\nNIFTY queue:", len(nifty_download_queue))


In [ ]:
# ============================================================
# 18. NIFTY PREFLIGHT
# ============================================================

nifty_existing_before_download = [
    day
    for day in nifty_download_queue
    if parquet_path(NIFTY_DIR, day).exists()
]

assert not nifty_existing_before_download, (
    "ABORT: NIFTY queue contains existing files: "
    f"{nifty_existing_before_download[:20]}"
)

print("NIFTY PREFLIGHT PASSED")
print("Queued missing files:", len(nifty_download_queue))


In [ ]:
# ============================================================
# 19. NIFTY CHUNK PLANNER
# ============================================================

def make_nifty_chunks(missing_dates, chunk_days=NIFTY_CHUNK_DAYS):
    if not missing_dates:
        return []

    start = min(missing_dates)
    end = max(missing_dates)

    chunks = []
    cur = start

    while cur <= end:
        chunk_end = min(
            cur + timedelta(days=chunk_days - 1),
            end,
        )
        chunks.append((cur, chunk_end))
        cur = chunk_end + timedelta(days=1)

    return chunks

nifty_chunks = make_nifty_chunks(nifty_download_queue)

print("NIFTY chunks:", len(nifty_chunks))

for x in nifty_chunks[:10]:
    print(" ", x)

if len(nifty_chunks) > 10:
    print(" ...")


In [ ]:
# ============================================================
# 20. NIFTY DOWNLOAD LOOP
# ============================================================
# IMPORTANT:
# We DO NOT make one API call per missing date.
# We fetch date ranges through the official NIFTY API and write
# only the missing dates.

nifty_results = []

if DOWNLOAD_NIFTY and nifty_download_queue:

    missing_set = set(nifty_download_queue)

    print(
        f"Downloading NIFTY data for {len(missing_set):,} missing dates "
        f"using {len(nifty_chunks):,} API chunks."
    )

    for chunk_no, (chunk_start, chunk_end) in enumerate(
        nifty_chunks,
        1,
    ):

        # Only dates from this chunk that are actually missing.
        chunk_missing = [
            day
            for day in nifty_download_queue
            if chunk_start <= day <= chunk_end
        ]

        if not chunk_missing:
            continue

        # HARD CHECK immediately before the chunk's network request.
        existing_now = [
            day
            for day in chunk_missing
            if parquet_path(NIFTY_DIR, day).exists()
        ]

        if existing_now:
            raise RuntimeError(
                "ABORT: NIFTY files appeared after preflight: "
                f"{existing_now[:20]}"
            )

        print(
            f"\nNIFTY chunk {chunk_no}/{len(nifty_chunks)}: "
            f"{chunk_start} -> {chunk_end} | "
            f"missing dates={len(chunk_missing):,}"
        )

        df = fetch_nifty_range(
            nifty_session,
            chunk_start,
            chunk_end,
            verbose=True,
        )

        if df.empty:
            # No data for this chunk. Do not fabricate files.
            for day in chunk_missing:
                result = {
                    "date": day,
                    "status": "no_data",
                    "rows": 0,
                    "chunk_start": chunk_start,
                    "chunk_end": chunk_end,
                    "error": "",
                }
                nifty_results.append(result)
                append_manifest(result, NIFTY_MANIFEST)
            continue

        returned_dates = set(df["date"])

        for day in chunk_missing:

            path = parquet_path(NIFTY_DIR, day)

            # HARD CHECK immediately before writing this date.
            if path.exists():
                result = {
                    "date": day,
                    "status": "skipped_existing",
                    "rows": 0,
                    "chunk_start": chunk_start,
                    "chunk_end": chunk_end,
                    "error": "",
                }
                nifty_results.append(result)
                append_manifest(result, NIFTY_MANIFEST)
                continue

            if day not in returned_dates:
                result = {
                    "date": day,
                    "status": "not_returned",
                    "rows": 0,
                    "chunk_start": chunk_start,
                    "chunk_end": chunk_end,
                    "error": "Date not present in API response",
                }
                nifty_results.append(result)
                append_manifest(result, NIFTY_MANIFEST)
                continue

            day_df = df[df["date"] == day].copy()

            # Ensure one expected NIFTY row.
            if day_df.empty:
                result = {
                    "date": day,
                    "status": "not_returned",
                    "rows": 0,
                    "chunk_start": chunk_start,
                    "chunk_end": chunk_end,
                    "error": "Empty filtered dataframe",
                }
                nifty_results.append(result)
                append_manifest(result, NIFTY_MANIFEST)
                continue

            # Convert date to timestamp for consistent Parquet schema.
            day_df["date"] = pd.Timestamp(day)

            # Last no-redownload check.
            if path.exists():
                result = {
                    "date": day,
                    "status": "skipped_existing",
                    "rows": 0,
                    "chunk_start": chunk_start,
                    "chunk_end": chunk_end,
                    "error": "",
                }
            else:
                atomic_to_parquet(day_df, path)

                result = {
                    "date": day,
                    "status": "downloaded",
                    "rows": len(day_df),
                    "chunk_start": chunk_start,
                    "chunk_end": chunk_end,
                    "error": "",
                }

            nifty_results.append(result)
            append_manifest(result, NIFTY_MANIFEST)

else:
    print("No missing NIFTY files. No NIFTY API requests will be made.")

nifty_results_df = pd.DataFrame(nifty_results)


In [ ]:
# ============================================================
# 21. FINAL VALIDATION
# ============================================================

def validate_sync(directory, expected_dates, name):
    missing = [
        day for day in expected_dates
        if not parquet_path(directory, day).exists()
    ]

    existing = [
        day for day in expected_dates
        if parquet_path(directory, day).exists()
    ]

    print(f"\n{name} FINAL FILESYSTEM VALIDATION")
    print("-" * 70)
    print("Expected dates:", len(expected_dates))
    print("Parquet files :", len(existing))
    print("Still missing :", len(missing))

    if missing:
        print("First missing:", missing[:50])

    return missing

stock_still_missing = validate_sync(
    STOCK_DIR,
    stock_candidates,
    "STOCKS",
)

nifty_still_missing = validate_sync(
    NIFTY_DIR,
    nifty_candidates,
    "NIFTY 50",
)


In [ ]:
# ============================================================
# 22. PARQUET CONTENT VALIDATION
# ============================================================

def validate_parquet_schema(directory, sample_dates, name):
    expected_cols = [
        "date",
        "symbol",
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]

    checked = 0

    for day in sample_dates:
        path = parquet_path(directory, day)

        if not path.exists():
            continue

        df = pd.read_parquet(path)

        if list(df.columns) != expected_cols:
            raise AssertionError(
                f"{name} schema mismatch for {path}: "
                f"{list(df.columns)}"
            )

        if df.empty:
            raise AssertionError(
                f"{name} empty parquet: {path}"
            )

        checked += 1

        if checked >= 20:
            break

    print(f"{name}: validated {checked} Parquet files.")

validate_parquet_schema(
    STOCK_DIR,
    stock_candidates,
    "STOCKS",
)

validate_parquet_schema(
    NIFTY_DIR,
    nifty_candidates,
    "NIFTY 50",
)


In [ ]:
# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("NSE STOCKS + NIFTY 50 SYNC COMPLETE")
print("=" * 80)

print("\nSTOCKS")
print("  Existing at start :", len(stock_existing))
print("  Missing at start  :", len(stock_missing))
print("  Download queue    :", len(stock_download_queue))
print("  Still missing     :", len(stock_still_missing))

if not stock_results_df.empty:
    print("  Run statuses:")
    print(stock_results_df["status"].value_counts().to_string())

print("\nNIFTY 50")
print("  Existing at start :", len(nifty_existing))
print("  Missing at start  :", len(nifty_missing))
print("  Download queue    :", len(nifty_download_queue))
print("  Still missing     :", len(nifty_still_missing))

if not nifty_results_df.empty:
    print("  Run statuses:")
    print(nifty_results_df["status"].value_counts().to_string())

print("\nPaths")
print("  Stocks:", STOCK_DIR)
print("  NIFTY :", NIFTY_DIR)

print("\nRERUN RULE")
print("  Existing Parquet -> NEVER request again")
print("  Missing Parquet  -> retry on next run")
print("  Manifest         -> diagnostic only")


## What happens on the next run?

Suppose 4,100 dates are in the overall date range:

- 3,900 already have Parquet files → **0 requests for those 3,900**
- 150 previously failed → **150 are retried**
- 50 have never been attempted → **50 are attempted**
- If a file appears between inventory and request → **the request is skipped**
- NIFTY is fetched through range API calls, **not 4,100 individual API calls**
- NIFTY API records outside the requested range are discarded before writing
- No manifest entry can force an existing Parquet file to be downloaded again.

The filesystem is the source of truth.
